# XGBoost baseline · `xgb_b_v1`

Huấn luyện 5 fold trên cùng `data/folds.csv` của nhóm, dùng 24 đặc trưng gốc và ROC AUC. Thiết lập này dùng số vòng cố định để giữ mỗi fold validation dành cho đánh giá. Chọn kernel `fraud-b` trong VS Code. Nếu cần CPU, đổi `DEVICE` thành `"cpu"` trước khi chạy lại.

Notebook lưu OOF và test prediction theo thứ tự CSV gốc. File submission theo thứ tự ID của sample submission.

In [1]:
import json
from hashlib import file_digest
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import xgboost as xgb
from IPython.display import display
from sklearn.metrics import roc_auc_score

EXP_ID = "xgb_b_v1"
DEVICE = "cuda"
N_FOLDS = 5
SEED = 42
N_ROUNDS = 600
ID = "id"
TARGET = "is_fraud"
CAT_FEATURES = ["merchant_category", "card_type", "auth_method", "channel", "device_type"]

PARAMS = dict(
    objective="binary:logistic",
    tree_method="hist",
    device=DEVICE,
    eval_metric="auc",
    n_estimators=N_ROUNDS,
    max_depth=6,
    learning_rate=0.04,
    min_child_weight=5,
    subsample=0.85,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    random_state=SEED,
    n_jobs=4,
    enable_categorical=True,
)

cwd = Path.cwd().resolve()
ROOT = next(
    (p for p in (cwd, *cwd.parents) if (p / "shared" / "make_folds.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Mở notebook từ thư mục repo Credit Fraud Detection.")

DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "B" / "outputs"
SUBMISSION_DIR = ROOT / "submissions"
print(f"{EXP_ID} | XGBoost {xgb.__version__} | pandas {pd.__version__} | {DEVICE}")

xgb_b_v1 | XGBoost 3.4.1 | pandas 3.1.1 | cuda


## Dữ liệu và fold

Ghép fold theo `id` bằng cách căn lại theo thứ tự train. So khớp tập ID test với sample submission và kiểm tra schema trước khi train.

In [2]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
folds = pd.read_csv(DATA_DIR / "folds.csv")
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")

for name, frame in [("train", train), ("test", test), ("folds", folds), ("sample", sample)]:
    if ID not in frame or frame[ID].isna().any() or not frame[ID].is_unique:
        raise ValueError(f"{name} cần có ID duy nhất và không thiếu.")

if list(folds.columns) != [ID, "fold"] or list(sample.columns) != [ID, TARGET]:
    raise ValueError("Schema yêu cầu: folds có id,fold và sample có id,is_fraud.")
if len(folds) != len(train) or not train[ID].isin(folds[ID]).all():
    raise ValueError("Tập ID của folds.csv và train.csv không khớp.")
if len(sample) != len(test) or not sample[ID].isin(test[ID]).all():
    raise ValueError("Tập ID của sample_submission.csv và test.csv không khớp.")

fold_ids = folds.set_index(ID)["fold"].reindex(train[ID])
if fold_ids.isna().any() or set(fold_ids) != set(range(N_FOLDS)):
    raise ValueError("folds.csv phải có đầy đủ fold 0–4 và không thiếu giá trị.")
fold_ids = fold_ids.to_numpy(dtype=np.int8)

if TARGET not in train or train[TARGET].isna().any() or set(train[TARGET]) != {0, 1}:
    raise ValueError("Train cần có nhãn is_fraud nhị phân, gồm cả hai lớp.")
y = train[TARGET].to_numpy(dtype=np.int8)
fold_summary = pd.crosstab(fold_ids, y, rownames=["fold"], colnames=[TARGET])
if (fold_summary == 0).any().any():
    raise ValueError("Mỗi fold cần có cả hai lớp để tính ROC AUC.")

features = [c for c in train if c not in (ID, TARGET, "fold")]
if len(features) != 24 or set(test) != {ID, *features} or not set(CAT_FEATURES).issubset(features):
    raise ValueError("Train/test phải có cùng 24 đặc trưng gốc.")
numeric_features = [c for c in features if c not in CAT_FEATURES]
X = train[features].astype({c: "float32" for c in numeric_features})
X_test = test[features].astype({c: "float32" for c in numeric_features})

with (DATA_DIR / "folds.csv").open("rb") as f:
    folds_sha256 = file_digest(f, "sha256").hexdigest()

print(f"Train: {len(train):,} | Test: {len(test):,} | Gian lận: {y.mean():.2%}")
print(f"Đặc trưng: {len(features)} | SHA-256 của folds.csv: {folds_sha256}")
display(fold_summary)

Train: 699,960 | Test: 299,984 | Gian lận: 1.67%
Đặc trưng: 24 | SHA-256 của folds.csv: 3dac00258735b21b169605d59587c17a6a02b7ce587f4545ad8d347038eaa6f4


is_fraud,0,1
fold,,
0,137655,2337
1,137654,2338
2,137654,2338
3,137654,2338
4,137654,2338


## Huấn luyện và dự đoán

Fit danh mục từ dữ liệu train của từng fold, rồi áp dụng cùng danh mục cho validation và test. Giá trị mới được biểu diễn là dữ liệu thiếu. Mỗi fold cho một dự đoán OOF trên phần validation và một dự đoán test. Dự đoán test cuối là trung bình của cả năm fold.

In [3]:
oof_pred = np.full(len(X), np.nan, dtype=np.float32)
test_pred_sum = np.zeros(len(X_test), dtype=np.float64)
fold_metrics = []
start_time = perf_counter()

for fold in range(N_FOLDS):
    fold_start = perf_counter()
    valid_mask = fold_ids == fold
    X_fit = X.loc[~valid_mask].copy()
    X_valid = X.loc[valid_mask].copy()
    X_predict = X_test.copy()

    for col in CAT_FEATURES:
        dtype = pd.CategoricalDtype(X_fit[col].dropna().unique())
        X_fit[col] = X_fit[col].astype(dtype)
        X_valid[col] = X_valid[col].astype(dtype)
        X_predict[col] = X_predict[col].astype(dtype)

    model = xgb.XGBClassifier(**PARAMS)
    model.fit(X_fit, y[~valid_mask])
    booster = model.get_booster()

    actual_device = json.loads(booster.save_config())["learner"]["generic_param"]["device"]
    if DEVICE == "cuda" and not actual_device.startswith("cuda"):
        raise RuntimeError(
            f"Yêu cầu CUDA nhưng mô hình đang dùng {actual_device}. "
            "Kiểm tra driver hoặc chọn DEVICE='cpu' rồi chạy lại."
        )

    valid_prob = booster.predict(xgb.DMatrix(X_valid, enable_categorical=True, nthread=4))
    test_prob = booster.predict(xgb.DMatrix(X_predict, enable_categorical=True, nthread=4))

    oof_pred[valid_mask] = valid_prob
    test_pred_sum += test_prob

    auc = float(roc_auc_score(y[valid_mask], valid_prob))
    seconds = round(perf_counter() - fold_start, 2)
    fold_metrics.append(
        {"fold": fold, "auc": auc, "n_valid": int(valid_mask.sum()), "seconds": seconds}
    )
    print(f"Fold {fold} | AUC {auc:.6f} | {actual_device} | {seconds:.1f}s")
    del model, booster, X_fit, X_valid, X_predict

test_pred = (test_pred_sum / N_FOLDS).astype(np.float32)
total_seconds = round(perf_counter() - start_time, 2)

Fold 0 | AUC 0.944431 | cuda:0 | 6.9s


Fold 1 | AUC 0.947836 | cuda:0 | 7.3s


Fold 2 | AUC 0.950701 | cuda:0 | 6.4s


Fold 3 | AUC 0.945807 | cuda:0 | 6.6s


Fold 4 | AUC 0.941384 | cuda:0 | 6.7s


## Tổng hợp CV và xác nhận artifact

`cv_auc` và `cv_std` lần lượt là trung bình và độ lệch chuẩn tổng thể của 5 AUC theo fold. AUC tính trên toàn bộ OOF được báo riêng. Để so sánh sau này, giữ lại file fold, dự đoán và JSON chỉ số của lần chạy.

In [4]:
for name, pred, length in [("OOF", oof_pred, len(train)), ("test", test_pred, len(test))]:
    if pred.shape != (length,) or not np.isfinite(pred).all() or np.any((pred < 0) | (pred > 1)):
        raise ValueError(f"Dự đoán {name} thiếu, sai kích thước hoặc ngoài [0, 1].")

scores = [row["auc"] for row in fold_metrics]
metrics = {
    "exp_id": EXP_ID,
    "cv_auc": float(np.mean(scores)),
    "cv_std": float(np.std(scores, ddof=0)),
    "oof_auc": float(roc_auc_score(y, oof_pred)),
    "folds": fold_metrics,
    "folds_sha256": folds_sha256,
    "n_train": len(train),
    "n_test": len(test),
    "features": features,
    "params": PARAMS,
    "device": actual_device,
    "xgboost_version": xgb.__version__,
    "training_seconds": total_seconds,
}
display(pd.DataFrame(fold_metrics))
print(f"CV AUC: {metrics['cv_auc']:.6f} ± {metrics['cv_std']:.6f}")
print(f"OOF AUC: {metrics['oof_auc']:.6f} | Tổng: {total_seconds:.1f}s")

submission = sample.copy()
submission[TARGET] = pd.Series(test_pred, index=test[ID]).reindex(sample[ID]).to_numpy()
if submission[TARGET].isna().any():
    raise ValueError("Không căn được dự đoán theo tất cả ID trong sample submission.")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
np.save(OUTPUT_DIR / f"{EXP_ID}_oof.npy", oof_pred)
np.save(OUTPUT_DIR / f"{EXP_ID}_test.npy", test_pred)
(OUTPUT_DIR / f"{EXP_ID}_metrics.json").write_text(
    json.dumps(metrics, indent=2) + "\n", encoding="utf-8"
)
submission.to_csv(SUBMISSION_DIR / f"{EXP_ID}.csv", index=False)
print(f"Đã lưu OOF, test prediction, metrics trong {OUTPUT_DIR.relative_to(ROOT)}.")
print(f"Submission: {SUBMISSION_DIR.relative_to(ROOT) / (EXP_ID + '.csv')}")

,fold,auc,n_valid,seconds
0,0,0.944431,139992,6.91
1,1,0.947836,139992,7.31
2,2,0.950701,139992,6.40
3,3,0.945807,139992,6.61
4,4,0.941384,139992,6.65


CV AUC: 0.946032 ± 0.003139
OOF AUC: 0.945983 | Tổng: 33.9s


Đã lưu OOF, test prediction, metrics trong B\outputs.
Submission: submissions\xgb_b_v1.csv
